# 🧠 Notebook 02 — NLP Pipeline & Skill Extraction
Run after 01_EDA.ipynb

In [ ]:
import os, sys
sys.path.append(os.path.join(os.getcwd(), '..'))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from src.skill_extractor import extract_skills, compute_skill_match, build_resume_profile
from src.resume_parser   import parse_resume
from src.matcher         import compute_match_score, generate_suggestions

print('✅ Modules loaded')

In [ ]:
df = pd.read_csv('../data/processed/cleaned_resumes.csv')
print(f'Loaded {len(df)} resumes')

In [ ]:
print('Extracting skills (may take 2 min)...')
results = []
for i, row in df.iterrows():
    r = extract_skills(str(row.get('resume','')))
    results.append({'category': row['category'], 'skill_count': r['skill_count'], 'top_domain': r['top_domain']})
df_skills = pd.DataFrame(results)
print('\n📊 Skill Stats:')
print(df_skills['skill_count'].describe().round(2))
print('\nTop domains:')
print(df_skills['top_domain'].value_counts().head(8))

In [ ]:
fig, axes = plt.subplots(1,2,figsize=(14,5))
axes[0].hist(df_skills['skill_count'], bins=30, color='#667eea', edgecolor='white')
axes[0].set_title('Skills Per Resume')
axes[0].set_xlabel('Number of Skills')
domain_counts = df_skills['top_domain'].value_counts().head(8)
axes[1].barh(domain_counts.index, domain_counts.values, color='#764ba2')
axes[1].set_title('Most Common Top Domain')
plt.tight_layout()
plt.savefig('../screenshots/plot5_skills.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
sample_resume = '''
Data Scientist with 3 years experience.
Python, SQL, Machine Learning, TensorFlow,
NLP, Docker, AWS, Git, Pandas, Flask
B.Tech Computer Science 2021
'''

sample_jd = '''
Hiring Data Scientist, 2+ years required.
Must have: Python, SQL, Machine Learning,
TensorFlow, NLP, Docker, AWS.
Nice to have: Spark, Kubernetes.
'''

result = compute_match_score(sample_resume, sample_jd)
print(f'Final Score      : {result["final_score_pct"]}%')
print(f'Verdict          : {result["verdict"]}')
print(f'TF-IDF Score     : {result["tfidf_score"]}%')
print(f'Skill Match      : {result["skill_score_pct"]}%')
print(f'Matched Skills   : {result["matched_skills"]}')
print(f'Missing Skills   : {result["missing_skills"]}')
for s in generate_suggestions(result):
    print(f'  Suggestion: {s}')

In [ ]:
scores = [result['tfidf_score'], result['skill_score_pct'], result['experience_score']]
labels = ['TF-IDF (50%)', 'Skills (35%)', 'Experience (15%)']
colors = ['#667eea','#28a745','#ffc107']
plt.figure(figsize=(8,4))
bars = plt.bar(labels, scores, color=colors, edgecolor='white', width=0.5)
for bar, val in zip(bars, scores):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
             f'{val:.1f}%', ha='center', fontsize=11, fontweight='bold')
plt.axhline(result['final_score_pct'], color='red', linestyle='--',
            label=f'Final Score: {result["final_score_pct"]}%')
plt.ylim(0, 110)
plt.title('Score Component Breakdown')
plt.legend()
plt.tight_layout()
plt.savefig('../screenshots/plot6_match.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ NLP notebook complete!')